In [22]:
import pandas as pd
import glob
import os

RAW_DIR = '../../data/norming_study/raw'

In [23]:
# load all half csvs (skip demographics)
half_files = glob.glob(os.path.join(RAW_DIR, '*_half.csv'))
raw = pd.concat([pd.read_csv(f) for f in half_files], ignore_index=True)

print(f'{len(half_files)} files loaded, {len(raw)} total rows')
raw.head()

3 files loaded, 19 total rows


,trial_type,subjectID,prolificID,studyID,sessionID,DEBUG,sliderOrder,half,itemID,actionPhrase,abilityResponse,abilityRT,abilityDragged,willingnessResponse,willingnessRT,willingnessDragged,trialRT,trialIndex,suspicious
0,whyask-trial,4dhnekfjvy,NaN,NaN,NaN,0,AW,1,98,hold something for someone for a moment,65,1081.0,False,61,1866.0,False,2309.0,1,False
1,whyask-trial,4dhnekfjvy,NaN,NaN,NaN,0,AW,1,78,help someone pick up something they dropped,73,787.0,True,59,1634.0,False,2520.0,2,False
2,whyask-trial,4dhnekfjvy,NaN,NaN,NaN,0,AW,1,37,let a classmate copy your exam answers,78,959.0,True,59,1748.0,True,2244.0,3,False
3,whyask-trial,4dhnekfjvy,NaN,NaN,NaN,0,AW,1,76,scoot over on a bench to make room,79,853.0,True,65,1529.0,True,2078.0,4,False
4,whyask-trial,4dhnekfjvy,NaN,NaN,NaN,0,AW,1,55,lend your car to an acquaintance for the weekend,81,560.0,False,64,1223.0,True,1617.0,5,False


In [24]:
# split trials vs attn checks
trials = raw[raw['half'].isin([1, 2])].copy()
attn   = raw[raw['half'] == 'attn'].copy()

print(f'trials: {len(trials)} rows, {trials["subjectID"].nunique()} subjects')
print(f'attn checks: {len(attn)} rows')
print(f'subjects: {sorted(trials["subjectID"].unique())}')

trials: 12 rows, 2 subjects
attn checks: 2 rows
subjects: ['4dhnekfjvy', 'xj0mmuh0bf']


In [25]:
# load demographics
demo_files = glob.glob(os.path.join(RAW_DIR, '*_demographics.csv'))
demo = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)

print(f'{len(demo)} demo rows')
demo[['subjectID', 'age', 'gender', 'race', 'education', 'totalDurationMs']]

1 demo rows


,subjectID,age,gender,race,education,totalDurationMs
0,xj0mmuh0bf,24,Female,Asian,bachelors,334154


In [26]:
# attn check summary — did they pass?
attn_cols = ['subjectID', 'itemID', 'abilityResponse', 'willingnessResponse', 'suspicious']
if 'targetValue' in attn.columns:
    attn_cols.insert(4, 'targetValue')
print(attn[attn_cols])



     subjectID  itemID  abilityResponse  willingnessResponse  suspicious
12  xj0mmuh0bf  ATTN_1               79                   79       False
13  xj0mmuh0bf  ATTN_2               43                   43       False


In [27]:
# subjects who failed any attn check (suspicious == True on any row)
failed = attn[attn['suspicious'] == True]['subjectID'].unique()
print(f'failed attn: {failed}')

trials_clean = trials[~trials['subjectID'].isin(failed)].copy()
demo_clean   = demo[~demo['subjectID'].isin(failed)].copy()

print(f'kept {trials_clean["subjectID"].nunique()} subjects ({len(trials_clean)} trials)')

failed attn: []
kept 2 subjects (12 trials)


In [30]:
# quick look at trial responses
print(trials[['subjectID', 'itemID', 'actionPhrase', 'abilityResponse', 'willingnessResponse', 'sliderOrder']].to_string())
print()
# avg time per trial
avg_rt = trials['trialRT'].mean() / 1000
print(f'avg trial RT: {avg_rt:.1f}s')

     subjectID itemID                                      actionPhrase  abilityResponse  willingnessResponse sliderOrder
0   4dhnekfjvy     98           hold something for someone for a moment               65                   61          AW
1   4dhnekfjvy     78       help someone pick up something they dropped               73                   59          AW
2   4dhnekfjvy     37            let a classmate copy your exam answers               78                   59          AW
3   4dhnekfjvy     76                scoot over on a bench to make room               79                   65          AW
4   4dhnekfjvy     55  lend your car to an acquaintance for the weekend               81                   64          AW
5   4dhnekfjvy     14                 sing a very high note in an opera               87                   66          AW
6   4dhnekfjvy     69                      tell someone what time it is               77                   61          AW
14  xj0mmuh0bf     89   

In [29]:
# save processed
PROC_DIR = '../../data/norming_study/processed'
os.makedirs(PROC_DIR, exist_ok=True)

# drop trial-level cols from demo
trial_cols = ['trialIndex', 'itemID', 'actionPhrase', 'abilityResponse', 'abilityRT',
              'abilityDragged', 'willingnessResponse', 'willingnessRT', 'willingnessDragged',
              'trialRT', 'suspicious', 'targetValue', 'passed']
demo_out = demo_clean.drop(columns=[c for c in trial_cols if c in demo_clean.columns])

trials_clean.to_csv(os.path.join(PROC_DIR, 'trials.csv'), index=False)
demo_out.to_csv(os.path.join(PROC_DIR, 'demographics.csv'), index=False)

print(f'saved trials.csv ({len(trials_clean)} rows) and demographics.csv ({len(demo_out)} rows)')
print(f'demo cols: {list(demo_out.columns)}')

saved trials.csv (12 rows) and demographics.csv (1 rows)
demo cols: ['trial_type', 'subjectID', 'prolificID', 'studyID', 'sessionID', 'DEBUG', 'sliderOrder', 'half', 'age', 'gender', 'race', 'education', 'strategy', 'technicalIssues', 'feedback', 'visibilityChanges', 'totalDurationMs']
